# NWB Example: IBL Brain Wide map

Data from Meshulam et al. (2025)¹ , a multi-lab dataset studying the neural basis of decision-making using neuropixel probes. This data set contains other information gathered during the task: sensory stimuli presented to the mouse; mouse decisions and response times; and mouse pose information from video recordings and DeepLabCut analysis.
 
- Link to entire dataset: https://neurosift.app/dandiset/000409
- Link to one subject session (downloaded here) https://neurosift.app/nwb?url=https://api.dandiarchive.org/api/assets/00cab48e-f9e2-497e-b1a8-cc19a019ee07/download/&dandisetId=000409 for subject CSHL045 from Churchland lab.


The code below shows how one can download and convert a single session `nwb` file (behavioural/pose data only) into the `TrialTree` format, as well as download some external videos linked to the file. 

**Note**: A significant amount of the code (in `ethograph.utils.nwb`) is dedicated to automatically identifying to identifying the video download links associated with this session usinig NWB `external_file` attribute, and then identifying which `nwb` PoseEstimationSeries data (see https://github.com/rly/ndx-pose) belong to which idea (e.g. Pose from CameraLeft to .mp4 of CameraLeft). Maybe this will be made easier in the future! See Git issue [here](https://github.com/int-brain-lab/iblenv/issues/432).

Note, that poes estimation data has to be `ndx-pose>v0.2.0`.


---
Meshulam, L., Angelaki, D., Benson, B., Benson, J., Birman, D., Arlandis, J., Bonacchi, N., Bougrova, K., Bruijns, S. A., Carandini, M., Catarino, J. A., Chapuis, G. A., Churchland, A. K., Dan, Y., Davatolhagh, F., Dayan, P., DeWitt, E. E., Engel, T. A., Fabbri, M., … Witten, I. B. (2025). A brain-wide map of neural activity during complex behaviour. Nature, 645(8079), 177–191. https://doi.org/10.1038/s41586-025-09235-0



In [3]:
%load_ext autoreload
%autoreload 2
import remfile
import h5py
import pynwb
from dandi.dandiapi import DandiAPIClient
from ethograph.utils.nwb import load_nwb_session, find_video_assets
from ethograph.labels.converters import NWBLabelConverter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
from dandi.dandiapi import DandiAPIClient
import remfile
import h5py
import pynwb
import numpy as np
import pandas as pd


def pose_to_napari_points(pose_series, start=0, stop=100):
    """Only extract points (not tracks) for computational efficiency."""
    all_points = []
    all_props = []

    keypoint_names = list(pose_series.keys())
    n_keypoints = len(keypoint_names)

    for kp_idx, keypoint in enumerate(keypoint_names):
        pes = pose_series[keypoint]

        pos = pes.data[start:stop]          # (T, 2)
        conf = (
            pes.confidence[start:stop]
            if getattr(pes, "confidence", None) is not None
            else np.ones(stop - start)
        )
        frames = np.arange(start, stop)

        points = np.column_stack([
            frames,
            pos[:, 1],  # y
            pos[:, 0],  # x
        ])
        all_points.append(points)

        all_props.append(pd.DataFrame({
            "keypoint":    keypoint,
            "keypoint_id": float(kp_idx) / max(n_keypoints - 1, 1),
            "confidence":  conf,
        }))

    return np.vstack(all_points), pd.concat(all_props, ignore_index=True)

# ----------------------------
# Get remote NWB file (lazy)
# ----------------------------
dandiset_id = "000409"
asset_id = "773516a9-bd20-4b46-adad-2a1d5772be5d"

with DandiAPIClient() as client:
    asset = client.get_dandiset(dandiset_id).get_asset(asset_id)
    url = asset.get_content_url(follow_redirects=1, strip_query=True)

# ----------------------------
# Open remote file lazily
# ----------------------------
rf = remfile.File(url)
h5_file = h5py.File(rf, "r")

# ----------------------------
# Load NWB structure (still lazy)
# ----------------------------
io = pynwb.NWBHDF5IO(file=h5_file, mode="r", load_namespaces=True)
nwb_file = io.read()

# ----------------------------
# Access pose estimation
# ----------------------------
pose_series = (
    nwb_file
    .processing["pose_estimation"]
    ["RightCamera"]
    .pose_estimation_series
)

# ----------------------------
# Convert ONLY a slice to napari
# ----------------------------
points, properties = pose_to_napari_points(
    pose_series,
    start=0,
    stop=100,   # 👈 only this range is fetched from remote
)

c:\Users\aksel\anaconda3\envs\ethograph\Lib\site-packages\hdmf\spec\namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
hdmf-common - cached version: 1.9.0, loaded version: 1.8.0
hdmf-experimental - cached version: 0.6.0, loaded version: 0.5.0
Please update to the latest package versions.
  self.warn_for_ignored_namespaces(ignored_namespaces)


(330, 3)

In [ ]:
import napari

viewer = napari.Viewer()

points, properties = pose_to_napari_points(pose_series, start=0, stop=100)



napari.run()

2026-04-03 17:22:10.732 | WARNING  | napari.layers.points.points:_update_thumbnail:1905 - c:\Users\aksel\anaconda3\envs\ethograph\Lib\site-packages\napari\layers\points\points.py:1905: RuntimeWarning: invalid value encountered in cast
  ).astype(int)

2026-04-03 17:22:10.733 | WARNING  | napari.layers.points.points:_update_thumbnail:1928 - c:\Users\aksel\anaconda3\envs\ethograph\Lib\site-packages\napari\layers\points\points.py:1928: RuntimeWarning: invalid value encountered in cast
  ).astype(int)



In [ ]:
path = r"C:\Users\aksel\Desktop\POPPY_LABEL\raw\behav\dlc - Copy\2026-03-08_001_Poppy-cam-1DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv"

from movement.io.load_poses import from_dlc_file


ds = from_dlc_file(path, fps=200)
ds

In [65]:
from movement.napari.convert import ds_to_napari_layers

x = ds_to_napari_layers(ds)

In [79]:
len(ds.keypoints)

22

In [80]:
x[0][:, 0]

array([ 0.,  0.,  0., ..., 21., 21., 21.], shape=(20328,))

In [88]:
import napari

viewer = napari.Viewer()

# Remove rows with NaN or Inf
mask = np.isfinite(tracks).all(axis=1)
tracks = tracks[mask]

# Apply same mask to properties
properties = properties.iloc[mask].reset_index(drop=True)

viewer.add_points(
    tracks[:, 1:], # skip track_id column
    properties=properties,
    name="pose_tracks",
)

napari.run()

In [20]:
right_camera

Data type,float64
Shape,"(573864, 2)"
Array size,8.76 MiB
Chunk shape,"(573864, 2)"
Compression,gzip
Compression opts,4
Uncompressed size (bytes),9181824
Compressed size (bytes),3755280
Compression ratio,2.445043778360069
Data type,float64
Shape,"(573864,)"


In [21]:
import pynwb
import lindi

local_cache = lindi.LocalCache()

# Load https://api.dandiarchive.org/api/assets/00cab48e-f9e2-497e-b1a8-cc19a019ee07/download/
f = lindi.LindiH5pyFile.from_hdf5_file("https://api.dandiarchive.org/api/assets/00cab48e-f9e2-497e-b1a8-cc19a019ee07/download/", local_cache=local_cache)
nwb = pynwb.NWBHDF5IO(file=f, mode='r').read()


### todo, extract skeleton from here for visualization 

In [ ]:
Skeletons = nwb.processing["pose_estimation"]["Skeletons"] 

In [22]:
LeftCamera = nwb.processing["pose_estimation"]["LeftCamera"]

In [ ]:
keypoints = LeftCamera.pose_estimation_series.keys()
keypoints

dict_keys(['PoseEstimationSeriesLeftPaw', 'PoseEstimationSeriesLeftTongueEnd', 'PoseEstimationSeriesNoseTip', 'PoseEstimationSeriesRightPaw', 'PoseEstimationSeriesRightPupilBottom', 'PoseEstimationSeriesRightPupilLeft', 'PoseEstimationSeriesRightPupilRight', 'PoseEstimationSeriesRightPupilTop', 'PoseEstimationSeriesRightTongueEnd', 'PoseEstimationSeriesTubeBottom', 'PoseEstimationSeriesTubeTop'])

In [41]:
import numpy as np
import pandas as pd

def _construct_properties_dataframe_from_slice_nwb(
    data: np.ndarray,
    individuals: np.ndarray,
    time: np.ndarray,
    keypoints: np.ndarray | None = None,
    confidence: np.ndarray | None = None,
) -> pd.DataFrame:
    """
    Construct a properties DataFrame from sliced data.

    Parameters
    ----------
    data
        Sliced pose data (not used directly except for length alignment).
    time
        Array of time/frame indices.
    keypoints
        Optional array of keypoint labels.
    confidence
        Optional confidence values. If None, defaults to 1.

    Returns
    -------
    pd.DataFrame
    """

    individuals = ["individual_0"]
    
    n = data.shape[0] if data.ndim > 0 else len(time)

    # Default confidence = 1 if not provided
    if confidence is None:
        confidence = np.ones(n)

    # Build base columns
    df_dict = {
        "individual": np.repeat(individuals, n // len(individuals)) if len(individuals) > 1 else np.full(n, individuals),
        "time": time[:n],
        "confidence": confidence[:n],
    }

    # Add keypoints if present
    if keypoints is not None:
        df_dict["keypoint"] = np.repeat(keypoints, n // len(keypoints)) if len(keypoints) > 1 else np.full(n, keypoints)

    df = pd.DataFrame(df_dict)

    # Order columns nicely
    order = ["individual"]
    if keypoints is not None:
        order.append("keypoint")
    order += ["time", "confidence"]

    return df[order]

In [42]:
import numpy as np
import pandas as pd

def pose_to_napari(
    pose_estimation_series,
    n_frames: int,
):
    """
    Convert NWB pose estimation series (per-keypoint) into napari Tracks format.

    Parameters
    ----------
    pose_estimation_series : dict
        Mapping: keypoint -> NWB TimeSeries
    n_frames : int
        Number of frames to include (use slicing upstream!)

    Returns
    -------
    tracks : np.ndarray
        Shape (N, 4): (track_id, frame, y, x)
    properties : pd.DataFrame
    """

    all_tracks = []
    all_props = []

    track_id = 0

    for keypoint, pes in pose_estimation_series.items():

        # Lazy slice (this is where remfile helps)
        pos = pes.data[:n_frames]  # shape (T, 2)
        
        # Handle confidence safely
        if hasattr(pes, "confidence") and pes.confidence is not None:
            conf = pes.confidence[:n_frames]
        else:
            conf = np.ones(n_frames)

        # Build tracks: (frame, y, x)
        frames = np.arange(n_frames)

        tracks = np.column_stack([
            np.full(n_frames, track_id),
            frames,
            pos[:, 1],  # y
            pos[:, 0],  # x
        ])

        all_tracks.append(tracks)

        # Properties
        df = pd.DataFrame({
            "keypoint": keypoint,
            "confidence": conf,
            "time": frames,
        })

        all_props.append(df)

        track_id += 1

    tracks_array = np.vstack(all_tracks)
    properties_df = pd.concat(all_props, ignore_index=True)

    return tracks_array, properties_df

In [ ]:
dandiset_id = "000409" 
asset_id = "773516a9-bd20-4b46-adad-2a1d5772be5d" # 
with DandiAPIClient() as client:
    asset = client.get_dandiset(dandiset_id).get_asset(asset_id)
    url = asset.get_content_url(follow_redirects=1, strip_query=True)


rf = remfile.File(url)
h5_file = h5py.File(rf, "r")
h5_file = h5_file[0:100]
io = pynwb.NWBHDF5IO(file=h5_file, mode="r", load_namespaces=True)
nwb_file = io.read()

from movement.io.load_poses import from_nwb_file


ds = from_nwb_file(nwb_file, processing_module_key="pose_estimation", pose_estimation_key="RightCamera")

In [6]:
from movement.io.load_poses import from_nwb_file


ds = from_nwb_file(nwb_file, processing_module_key="pose_estimation", pose_estimation_key="RightCamera")

In [21]:
registry = get_pose_camera_registry(nwb_file)
dandi_assets = find_video_assets(dandiset_id, nwb_file, asset_id)
match_dandi_videos(registry, dandi_assets, asset_id)

print(f"Video & Pose registry: {registry}")

Matched by camera key: LeftCamera -> https://api.dandiarchive.org/api/assets/e2d4561a-e0ec-4b89-aad0-acc246e9aa46/download/
Matched by camera key: RightCamera -> https://api.dandiarchive.org/api/assets/e8d5d62d-fb20-44e8-b8b4-a210f3953580/download/
Video & Pose registry: {'LeftCamera': {'processing_module_key': 'pose_estimation', 'pose_interface_key': 'LeftCamera', 'video_url': 'https://api.dandiarchive.org/api/assets/e2d4561a-e0ec-4b89-aad0-acc246e9aa46/download/'}, 'RightCamera': {'processing_module_key': 'pose_estimation', 'pose_interface_key': 'RightCamera', 'video_url': 'https://api.dandiarchive.org/api/assets/e8d5d62d-fb20-44e8-b8b4-a210f3953580/download/'}}


In [ ]:
dt, trials_df = load_nwb_session(nwb_file, registry, load_nwb_session=True)
label_converter = NWBLabelConverter()
label_dt = label_converter.from_nwb(nwb_file, trials_df)

### Video and pose don't data don't match?

In [8]:
import cv2
from typing import Any
from pathlib import Path
from dandi.dandiapi import DandiAPIClient


def find_video_assets(
    dandiset_id: str,
    nwb: Any,
) -> list[tuple[str, str]]:
    session_id = getattr(nwb, "session_id", None)
    identifier = getattr(nwb, "identifier", None)
    subject_id = getattr(getattr(nwb, "subject", None), "subject_id", None)

    search_terms = [t for t in [session_id, identifier[:8] if identifier else None, subject_id] if t]
    if not search_terms:
        return []

    with DandiAPIClient() as client:
        dandiset = client.get_dandiset(dandiset_id)
        video_assets = [
            (Path(asset.path).stem, f"https://api.dandiarchive.org/api/assets/{asset.identifier}/download/")
            for asset in dandiset.get_assets()
            if asset.path.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
            and any(term in asset.path for term in search_terms)
        ]

    return video_assets

dandiset_id = "000231" # 000409
with DandiAPIClient() as client:
    asset = client.get_dandiset(dandiset_id).get_asset(
        "ead24fb9-0d16-414a-9f40-0c8a7bd5ee2c"
    )
    url = asset.get_content_url(follow_redirects=1, strip_query=True)

# remfile — better h5py compatibility, recommended for movement/behavior converters
rf = remfile.File(url)
h5_file = h5py.File(rf, "r")
io = pynwb.NWBHDF5IO(file=h5_file, mode="r", load_namespaces=True)


nwb_file = io.read()


video_assets = find_video_assets(dandiset_id, nwb_file)



2026-03-06 08:36:32.806 | WARNING  | hdmf.spec.namespace:warn_for_ignored_namespaces:620 - c:\Users\aksel\anaconda3\envs\ethograph\Lib\site-packages\hdmf\spec\namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
ndx-pose - cached version: 0.1.1, loaded version: 0.2.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)

